# Hunyuan3D-2：正面圖自動補多視角，再生成 3D（Colab T4）

把未使用的正面圖放進 Google Drive 的 `DRIVE_ROOT/in`，例如 `character-novice_female-front.jpg`。流程會自動產生 `used-character-novice_female-left.jpg`、`-back.jpg`、`-right.jpg`，並**停在視角檢查**；確認滿意後，再手動執行形狀、貼圖和下載區塊。

**固定執行環境（必做）**：「執行階段 → 變更執行階段類型」選 **T4 GPU**，並把「執行階段版本」設為 **2026.07**（Python 3.12.13、PyTorch 2.11.0、NumPy 2.0.2）。Colab 預設的最新版已改用 Python 3.13，部分 3D 套件沒有對應 wheel，區塊 1 會擋下不符的版本。

**每次新開 Colab**：依序跑區塊 1、2、3。新 runtime 第一次安裝後會自動重新啟動；之後再跑 1、2、3。模型權重與編譯好的 CUDA 擴充都快取在 Drive，第二次起不需重新下載或重新編譯。

| 區塊 | 做什麼 | 注意 |
|---|---|---|
| 1 | 檢查 runtime 版本、GPU、記憶體、硬碟 | 必須是 runtime 2026.07 + T4 |
| 2 | 設定模型、Drive 路徑、提示詞、種子與面數 | 執行前檢查 Drive 空間 |
| 3 | 安裝固定版本套件、Hunyuan3D-2 和 MV-Adapter | 新 runtime 約 3～5 分鐘（首次編譯約 5～10 分鐘），完成後自動重啟 |
| 4 | 從 `in` 找未使用的正面圖 | 只讀取 `<類別>-<id>-front.jpg/png/webp`，忽略 `used-` 檔案 |
| 5 | 生成並顯示 front/left/back/right | **在這裡停下檢查**；換 `VIEW_SEED` 後重跑可重新生成 |
| 6 | 多視角形狀 | 只在你確認視角後手動執行；模型使用 front/left/back，right 僅供檢視 |
| 7 | 貼圖 | 以去背後的正面圖作貼圖條件 |
| 8 | 存 GLB 到 `out`、下載 zip、標記來源 | GLB 已存 Drive 後才將 front 改名為 `used-...-front` |

**模型**：MV-Adapter SDXL（以 fp16 載入，避免 Colab 12.7 GB 系統記憶體不足）。區塊 2 預設 `VIEW_COUNT = 4`，較適合 T4 顯存；想用官方 6 視角可改成 6（約需 14 GB VRAM）。Hunyuan3D-2mv 使用三視角做形狀，生成的 right 保留供人工檢查。

**Drive 空間**：模型快取放在 `DRIVE_ROOT/hf_cache`，估計約 22 GB；區塊 2 會在下載前檢查可用量。

若 Hugging Face 回報無權限，請在 Colab Secrets 設定 `HF_TOKEN`。模型授權依 MV-Adapter、SDXL 及 Tencent Hunyuan3D-2 Community License 各自條款為準。

In [ ]:
# @title 1. Check runtime version, GPU, RAM and disk
import os, platform, shutil, subprocess, sys, torch

# Pinned Colab runtime: Runtime > Change runtime type > Runtime version > 2026.07.
EXPECTED_PYTHON = (3, 12)
EXPECTED_TORCH = "2.11."
print(f"Python: {platform.python_version()}")
print(f"Torch : {torch.__version__}  CUDA {torch.version.cuda}")
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook is pinned to Colab runtime version 2026.07 (Python 3.12). "
        "Runtime > Change runtime type > Runtime version > 2026.07, then run block 1 again.")
if not torch.__version__.startswith(EXPECTED_TORCH):
    print(f"WARNING: runtime 2026.07 ships torch {EXPECTED_TORCH}x; this runtime has {torch.__version__}. Pinned packages may not match.")

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
props = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability(0)
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gb = shutil.disk_usage("/content").free / 1024**3
print(f"GPU   : {props.name}  VRAM {props.total_memory / 1024**3:.1f} GB  sm_{cap[0]}{cap[1]}")
print(f"RAM   : {ram_gb:.1f} GB")
print(f"Disk  : {disk_gb:.0f} GB free")
nvcc = shutil.which("nvcc")
print(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1] if nvcc else "nvcc: not found")
if props.total_memory / 1024**3 < 14:
    print("WARNING: less than 14 GB VRAM; set VIEW_COUNT = 4, keep LOW_VRAM = True and RENDER_SIZE = 1024.")

In [ ]:
# @title 2. Settings
# @markdown Model weights stay in Drive (about 22 GB). Compiled CUDA extensions are cached in Drive too.
USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/RagnarokChange/hunyuan3d"  # @param {type:"string"}
CACHE_MODELS_ON_DRIVE = True  # @param {type:"boolean"}
# Default 4 views (front/left/back/right) fits T4 VRAM more safely; 6 is the official MV-Adapter setting.
VIEW_COUNT = 4  # @param [4, 6] {type:"raw"}
VIEW_BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
VIEW_VAE_MODEL = "madebyollin/sdxl-vae-fp16-fix"
VIEW_ADAPTER_REPO = "huanngzh/mv-adapter"
VIEW_ADAPTER_COMMIT = "4277e0018232bac82bb2c103caf0893cedb711be"
HUNYUAN_COMMIT = "f8db63096c8282cb27354314d896feba5ba6ff8a"
VIEW_SIZE = 768
VIEW_STEPS = 30
VIEW_GUIDANCE = 3.0
VIEW_SEED = 1234  # @param {type:"integer"}
VIEW_REFERENCE_SCALE = 1.0
VIEW_PROMPT = "same subject as the reference image, preserve its category, silhouette, colors and distinctive details, orthographic multi-view turnaround, plain neutral background, no text"
VIEW_NEGATIVE_PROMPT = "watermark, low quality, deformed, inconsistent design, extra objects, text"
VIEW_AZIMUTH_DEG = [0, 45, 90, 180, 270, 315] if VIEW_COUNT == 6 else [0, 90, 180, 270]
# MV-Adapter azimuth 90 looks at the subject's left side, matching Hunyuan3D-2mv's "left" view.
VIEW_OUTPUT_INDICES = {view: VIEW_AZIMUTH_DEG.index(angle) for view, angle in (("left", 90), ("back", 180), ("right", 270))}

# Hunyuan3D-2mv conditions on front, left, and back. Right is generated for review.
SHAPE_MODEL = "hunyuan3d-dit-v2-mv-turbo"  # @param ["hunyuan3d-dit-v2-mv", "hunyuan3d-dit-v2-mv-turbo"]
SHAPE_STEPS = 5 if "turbo" in SHAPE_MODEL else 50
SHAPE_FLASHVDM = True  # @param {type:"boolean"}
PAINT_MODEL = "hunyuan3d-paint-v2-0"  # @param ["hunyuan3d-paint-v2-0", "hunyuan3d-paint-v2-0-turbo"]
LOW_VRAM = True  # @param {type:"boolean"}
TEXTURE_SIZE = 2048  # @param [1024, 2048] {type:"raw"}
RENDER_SIZE = 1024  # @param [1024, 2048] {type:"raw"}
SEED = 1234  # @param {type:"integer"}
ESTIMATED_CACHE_GB = 22

import os
import shutil

assert USE_DRIVE, "This workflow reads and archives assets in Google Drive."
from google.colab import drive
drive.mount("/content/drive")

INPUT_DIR = os.path.join(DRIVE_ROOT, "in")
OUT_DRIVE_DIR = os.path.join(DRIVE_ROOT, "out")
WHEEL_CACHE = os.path.join(DRIVE_ROOT, "wheels")
HF_HOME = os.path.join(DRIVE_ROOT, "hf_cache") if CACHE_MODELS_ON_DRIVE else "/content/hf_cache"
REMBG_HOME = os.path.join(DRIVE_ROOT, "rembg_cache") if CACHE_MODELS_ON_DRIVE else "/content/rembg_cache"
os.environ["HF_HOME"] = HF_HOME
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["U2NET_HOME"] = REMBG_HOME
os.environ["REMBG_HOME"] = REMBG_HOME
for directory in (INPUT_DIR, OUT_DRIVE_DIR, WHEEL_CACHE, HF_HOME, REMBG_HOME):
    os.makedirs(directory, exist_ok=True)

WORK = "/content/work"
SHAPE_DIR = os.path.join(WORK, "shape")
OUT_DIR = os.path.join(WORK, "out")
VIEW_DIR = INPUT_DIR
for directory in (SHAPE_DIR, OUT_DIR):
    os.makedirs(directory, exist_ok=True)

# Count only this workflow's repositories, deduplicating Hugging Face blob links.
MODEL_CACHE_REPOS = [VIEW_ADAPTER_REPO, VIEW_BASE_MODEL, VIEW_VAE_MODEL, "tencent/Hunyuan3D-2", "tencent/Hunyuan3D-2mv"]
seen_blobs = set()
cache_bytes = 0
for repo_id in MODEL_CACHE_REPOS:
    repo_cache = os.path.join(HF_HOME, "hub", "models--" + repo_id.replace("/", "--"))
    for root, _, names in os.walk(repo_cache):
        for name in names:
            path = os.path.join(root, name)
            try:
                stat = os.stat(path)
            except OSError:
                continue
            identity = (stat.st_dev, stat.st_ino)
            if identity not in seen_blobs:
                seen_blobs.add(identity)
                cache_bytes += stat.st_size
cache_gb = cache_bytes / 1024**3
cache_volume = DRIVE_ROOT if CACHE_MODELS_ON_DRIVE else HF_HOME
cache_volume_free_gb = shutil.disk_usage(cache_volume).free / 1024**3
needed_gb = max(0, ESTIMATED_CACHE_GB - cache_gb)
print(f"Cache volume free: {cache_volume_free_gb:.1f} GB | relevant cached models: {cache_gb:.1f} GB | estimated additional need: {needed_gb:.1f} GB")
if cache_volume_free_gb < needed_gb:
    raise RuntimeError(f"Not enough storage for the selected models. Free {needed_gb - cache_volume_free_gb:.1f} GB in Drive.")

PRESETS = {
    "character":   {"max_faces": 60000,  "octree": 384},
    "monster":     {"max_faces": 40000,  "octree": 384},
    "equipment":   {"max_faces": 15000,  "octree": 384},
    "prop":        {"max_faces": 15000,  "octree": 384},
    "environment": {"max_faces": 100000, "octree": 512},
}
DEFAULT_CATEGORY = "prop"
print("Settings ready. HF_HOME =", os.environ["HF_HOME"])

In [ ]:
# @title 3. Install (do not stop it while it runs)
# @markdown A new runtime installs pinned packages, reuses CUDA extensions cached in Drive, checks the pipelines, then restarts.
# @markdown After restart, run blocks 1, 2, 3 again; the marked install finishes in seconds.
import glob, os, shutil, subprocess, sys, sysconfig

REPO = "/content/Hunyuan3D-2"
MV_REPO = "/content/MV-Adapter"
MARK = "/content/.hy3d_mv_installed_v2"
CONSTRAINTS = "/content/constraints.txt"
# Versions verified against Colab runtime 2026.07 (Python 3.12, torch 2.11, numpy 2.0.2).
PINNED = [
    "transformers==4.57.3", "diffusers==0.34.0", "huggingface_hub==0.36.0", "accelerate==1.10.1",
    "peft==0.17.1", "omegaconf==2.3.0", "einops==0.8.1", "pymeshlab==2025.7.post1", "xatlas==0.0.11",
    "pygltflib==1.16.5", "trimesh==4.8.3", "onnxruntime==1.22.1", "pybind11==3.0.1", "ninja==1.13.0",
    "pooch==1.8.2", "jsonschema==4.25.1",
]
PINNED_NO_DEPS = ["rembg==2.0.67", "pymatting==1.1.14"]
NVDIFFRAST = "git+https://github.com/NVlabs/nvdiffrast.git@v0.3.3"


def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    process = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in process.stdout:
        tail = (tail + [line])[-40:]
        if not line.startswith(("  ", "Requirement already")):
            print(line, end="", flush=True)
    if process.wait() != 0:
        print("".join(tail))
        raise RuntimeError(f"Command failed: {cmd}")


def check_in_subprocess():
    code = ("import sys; sys.path[:0] = [%r, %r]; import numpy, scipy.spatial, torch; "
            "import custom_rasterizer, hy3dgen.shapegen, hy3dgen.texgen, rembg; "
            "from mvadapter.pipelines.pipeline_mvadapter_i2mv_sdxl import MVAdapterI2MVSDXLPipeline; "
            "from scripts.inference_i2mv_sdxl import run_pipeline; "
            "print('numpy', numpy.__version__, 'torch', torch.__version__, 'pipelines OK')"
            % (MV_REPO, REPO))
    result = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    print(result.stdout, result.stderr[-3000:])
    return result.returncode == 0


if not os.path.exists(MARK):
    import torch
    capability = torch.cuda.get_device_capability(0)
    abi_tag = f"py{sys.version_info[0]}{sys.version_info[1]}-torch{torch.__version__.replace('+', '_')}-sm{capability[0]}{capability[1]}"
    wheel_dir = os.path.join(WHEEL_CACHE, abi_tag)
    os.makedirs(wheel_dir, exist_ok=True)

    # Keep Colab's compiled stack; a different torch or numpy would break CUDA extensions.
    frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    keep = ("numpy==", "scipy==", "torch==", "torchvision==", "triton==", "numba==", "llvmlite==",
            "pillow==", "opencv-python==", "opencv-python-headless==", "scikit-image==")
    with open(CONSTRAINTS, "w") as file:
        file.write("".join(line + "\n" for line in frozen.splitlines() if line.lower().startswith(keep)))
    print(open(CONSTRAINTS).read())

    if not os.path.exists(REPO):
        sh(f"git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git {REPO}")
    sh(f"git checkout -q {HUNYUAN_COMMIT}", cwd=REPO)
    if not os.path.exists(MV_REPO):
        sh(f"git clone https://github.com/huanngzh/MV-Adapter.git {MV_REPO}")
    sh(f"git checkout -q {VIEW_ADAPTER_COMMIT}", cwd=MV_REPO)

    sh(f"pip install -c {CONSTRAINTS} " + " ".join(f"'{package}'" for package in PINNED))
    sh(f"pip install -c {CONSTRAINTS} --no-deps " + " ".join(PINNED_NO_DEPS))
    sh(f"pip install --no-deps -e {REPO}")
    # nvdiffrast 0.3.x compiles lazily and is only imported by MV-Adapter, so this install is quick.
    sh(f"pip install --no-build-isolation --no-deps {NVDIFFRAST}")

    # custom_rasterizer: build once per Python/torch/GPU combination, then reuse the wheel from Drive.
    wheels = sorted(glob.glob(os.path.join(wheel_dir, "custom_rasterizer-*.whl")))
    if not wheels:
        os.environ["TORCH_CUDA_ARCH_LIST"] = f"{capability[0]}.{capability[1]}"
        build_dir = "/content/wheel_build"
        shutil.rmtree(build_dir, ignore_errors=True)
        sh(f"pip wheel -c {CONSTRAINTS} --no-build-isolation --no-deps -w {build_dir} .",
           cwd=f"{REPO}/hy3dgen/texgen/custom_rasterizer")
        for wheel in glob.glob(os.path.join(build_dir, "custom_rasterizer-*.whl")):
            shutil.copy2(wheel, wheel_dir)
        wheels = sorted(glob.glob(os.path.join(wheel_dir, "custom_rasterizer-*.whl")))
    else:
        print("Using cached wheel:", wheels[-1])
    sh(f"pip install --no-deps --force-reinstall '{wheels[-1]}'")

    renderer_dir = f"{REPO}/hy3dgen/texgen/differentiable_renderer"
    mesh_processor = f"mesh_processor{sysconfig.get_config_var('EXT_SUFFIX')}"
    cached_processor = os.path.join(wheel_dir, mesh_processor)
    if not os.path.exists(cached_processor):
        try:
            sh(f"c++ -O3 -Wall -shared -std=c++11 -fPIC $({sys.executable} -m pybind11 --includes) "
               f"mesh_processor.cpp -o {mesh_processor}", cwd=renderer_dir)
            shutil.copy2(os.path.join(renderer_dir, mesh_processor), cached_processor)
        except RuntimeError:
            print("mesh_processor build failed; using the slower Python fallback.")
    else:
        print("Using cached", cached_processor)
        shutil.copy2(cached_processor, renderer_dir)

    if not check_in_subprocess():
        raise RuntimeError("Pipeline import check failed. Send the last 30 lines above for diagnosis.")
    open(MARK, "w").close()
    print("\nInstall OK. Restarting the runtime; then run blocks 1, 2, 3 again.")
    os.kill(os.getpid(), 9)

for path in (REPO, MV_REPO):
    if path not in sys.path:
        sys.path.insert(0, path)
import custom_rasterizer  # noqa: F401
import hy3dgen.shapegen, hy3dgen.texgen  # noqa: F401
from mvadapter.pipelines.pipeline_mvadapter_i2mv_sdxl import MVAdapterI2MVSDXLPipeline
from mvadapter.schedulers.scheduling_shift_snr import ShiftSNRScheduler
from scripts.inference_i2mv_sdxl import run_pipeline as run_mv_sdxl
print("Install OK: Hunyuan3D-2mv and MV-Adapter SDXL are available.")

In [ ]:
# @title 4. Add images
# @markdown Put files such as character-novice_female-front.jpg in DRIVE_ROOT/in. Files starting with used- are ignored.
SOURCE = "drive"  # @param ["drive", "upload"]
import glob
import os
from PIL import Image

if SOURCE == "upload":
    from google.colab import files
    for name, data in files.upload().items():
        target = os.path.join(INPUT_DIR, os.path.basename(name))
        with open(target, "wb") as file:
            file.write(data)


def parse_name(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    lowered = stem.lower()
    if lowered.startswith("used-") or not lowered.endswith("-front"):
        return None
    for category in PRESETS:
        prefix = category + "-"
        if lowered.startswith(prefix):
            asset_id = stem[len(prefix):-len("-front")]
            if asset_id:
                return category, asset_id.lower()
    return None


image_paths = sorted(
    path for path in glob.glob(os.path.join(INPUT_DIR, "*"))
    if path.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
)
ASSETS = []
seen_keys = set()
for path in image_paths:
    parsed = parse_name(path)
    if parsed is None:
        continue
    category, asset_id = parsed
    key = f"{category}-{asset_id}"
    if key in seen_keys:
        raise ValueError(f"Duplicate front image for {key}; keep only one source image.")
    seen_keys.add(key)
    image = Image.open(path)
    alpha = image.mode == "RGBA" and image.getextrema()[3][0] < 255
    ASSETS.append({"category": category, "asset_id": asset_id, "key": key, "source": path})
    print(f"{os.path.basename(path):50s} {image.width}x{image.height}  alpha={'yes' if alpha else 'no (will remove)'}")

if not ASSETS:
    raise FileNotFoundError(f"No unused <category>-<id>-front image found in {INPUT_DIR}.")
print(f"{len(ASSETS)} front image(s) ready for multiview generation.")

In [ ]:
# @title 5. Generate and review multiviews
# @markdown This cell only generates and displays images. Inspect front/left/back/right here; change VIEW_SEED and rerun this cell to try another set. Do not run block 6 until satisfied.
import gc
import os
import torch
from diffusers import AutoencoderKL
from IPython.display import display
from PIL import Image, ImageOps
from hy3dgen.rembg import BackgroundRemover

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

free_vram_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Free GPU memory before view generation: {free_vram_gb:.1f} GB")
if VIEW_COUNT == 6 and free_vram_gb < 14:
    print("WARNING: MV-Adapter SDXL with 6 views needs about 14 GB VRAM. If this cell runs out of memory, set VIEW_COUNT = 4 in block 2.")

references = {}
background_remover = None
for asset in ASSETS:
    image = ImageOps.exif_transpose(Image.open(asset["source"]))
    has_alpha = image.mode == "RGBA" and image.getchannel("A").getextrema()[0] < 255
    if has_alpha:
        references[asset["key"]] = image.convert("RGBA")
    else:
        if background_remover is None:
            background_remover = BackgroundRemover()
        references[asset["key"]] = background_remover(image.convert("RGB"))
    # Block 6 and 7 reuse this exact cutout so every stage sees the same front image.
    references[asset["key"]].save(os.path.join(VIEW_DIR, f"used-{asset['key']}-front-cutout.png"))

del background_remover
gc.collect()
torch.cuda.empty_cache()

# Load SDXL directly in fp16: the upstream helper loads fp32 first, which exceeds free Colab's 12.7 GB RAM.
view_pipe = MVAdapterI2MVSDXLPipeline.from_pretrained(
    VIEW_BASE_MODEL,
    vae=AutoencoderKL.from_pretrained(VIEW_VAE_MODEL, torch_dtype=torch.float16),
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
view_pipe.scheduler = ShiftSNRScheduler.from_scheduler(
    view_pipe.scheduler, shift_mode="interpolated", shift_scale=8.0, scheduler_class=None)
view_pipe.init_custom_adapter(num_views=len(VIEW_AZIMUTH_DEG))
view_pipe.load_custom_adapter(VIEW_ADAPTER_REPO, weight_name="mvadapter_i2mv_sdxl.safetensors")
view_pipe.to(device="cuda", dtype=torch.float16)
view_pipe.cond_encoder.to(device="cuda", dtype=torch.float16)
view_pipe.enable_vae_slicing()

for asset in ASSETS:
    generated, _ = run_mv_sdxl(
        view_pipe,
        num_views=len(VIEW_AZIMUTH_DEG),
        text=VIEW_PROMPT,
        image=references[asset["key"]],
        height=VIEW_SIZE,
        width=VIEW_SIZE,
        num_inference_steps=VIEW_STEPS,
        guidance_scale=VIEW_GUIDANCE,
        seed=VIEW_SEED,
        reference_conditioning_scale=VIEW_REFERENCE_SCALE,
        negative_prompt=VIEW_NEGATIVE_PROMPT,
        device="cuda",
        azimuth_deg=VIEW_AZIMUTH_DEG,
    )
    if len(generated) != len(VIEW_AZIMUTH_DEG):
        raise RuntimeError(f"Expected {len(VIEW_AZIMUTH_DEG)} camera views for {asset['key']}, received {len(generated)}.")

    print(f"{asset['key']} | seed={VIEW_SEED} | inspect all four before continuing (left should face image-left)")
    display(references[asset["key"]])
    for view, index in VIEW_OUTPUT_INDICES.items():
        output_path = os.path.join(VIEW_DIR, f"used-{asset['key']}-{view}.jpg")
        generated[index].convert("RGB").save(output_path, quality=95)
        print(f"{view}: {output_path}")
        display(generated[index])
    torch.cuda.empty_cache()

del view_pipe, references
gc.collect()
torch.cuda.empty_cache()
print("Review complete. Rerun this cell with a new VIEW_SEED if any view needs another attempt.")

In [ ]:
# @title 6. Stage 1: multiview shapes
# @markdown Run only after approving the four images in block 5. The shape model uses front/left/back; right is retained for review.
import gc
import os
import time
import torch
from PIL import Image
from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import (Hunyuan3DDiTFlowMatchingPipeline, FloaterRemover,
                              DegenerateFaceRemover, FaceReducer)

_rembg = None


def prepare_view(path):
    global _rembg
    image = Image.open(path)
    if image.mode == "RGBA" and image.getchannel("A").getextrema()[0] < 255:
        return image
    if _rembg is None:
        _rembg = BackgroundRemover()
    return _rembg(image.convert("RGB"))


shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    "tencent/Hunyuan3D-2mv",
    subfolder=SHAPE_MODEL,
    variant="fp16",
    dtype=torch.float16,
)
if SHAPE_FLASHVDM and "turbo" in SHAPE_MODEL:
    # Official turbo setting: swaps in the turbo VAE and a faster decoder.
    shape_pipe.enable_flashvdm()

for asset in ASSETS:
    key = asset["key"]
    preset = PRESETS[asset["category"]]
    view_paths = {
        "front": os.path.join(VIEW_DIR, f"used-{key}-front-cutout.png"),
        "left": os.path.join(VIEW_DIR, f"used-{key}-left.jpg"),
        "back": os.path.join(VIEW_DIR, f"used-{key}-back.jpg"),
        "right": os.path.join(VIEW_DIR, f"used-{key}-right.jpg"),
    }
    missing = [view for view, path in view_paths.items() if not os.path.exists(path)]
    if missing:
        raise FileNotFoundError(f"{key} is missing approved view image(s): {', '.join(missing)}. Run block 5 first.")

    out_shape = os.path.join(SHAPE_DIR, f"{key}.glb")
    newest_view = max(os.path.getmtime(path) for path in view_paths.values())
    if os.path.exists(out_shape) and os.path.getmtime(out_shape) >= newest_view:
        print("skip (shape is current):", out_shape)
        continue

    t0 = time.time()
    images = {
        view: prepare_view(path)
        for view, path in view_paths.items()
        if view in ("front", "left", "back")
    }
    images["front"].save(os.path.join(SHAPE_DIR, f"{key}-input.png"))
    mesh = shape_pipe(
        image=images,
        num_inference_steps=SHAPE_STEPS,
        guidance_scale=5.0,
        octree_resolution=preset["octree"],
        num_chunks=8000 if preset["octree"] <= 384 else 4000,
        generator=torch.Generator(device="cuda").manual_seed(SEED),
        output_type="trimesh",
    )[0]
    raw_faces = len(mesh.faces)
    mesh = FloaterRemover()(mesh)
    mesh = DegenerateFaceRemover()(mesh)
    mesh = FaceReducer()(mesh, max_facenum=preset["max_faces"])
    mesh.export(out_shape)
    print(f"{key}: {raw_faces:,} -> {len(mesh.faces):,} faces ({time.time() - t0:.0f} s) -> {out_shape}")
    torch.cuda.empty_cache()

del shape_pipe
_rembg = None
gc.collect()
torch.cuda.empty_cache()
print("Stage 1 done. VRAM in use:", f"{torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
# @title 7. Stage 2: textures
import gc
import json
import os
import time
import torch
import trimesh
from PIL import Image
from hy3dgen.texgen import Hunyuan3DPaintPipeline

paint = Hunyuan3DPaintPipeline.from_pretrained("tencent/Hunyuan3D-2", subfolder=PAINT_MODEL)
paint.config.texture_size = TEXTURE_SIZE
paint.config.render_size = RENDER_SIZE
paint.render.set_default_texture_resolution(TEXTURE_SIZE)
paint.render.set_default_render_resolution(RENDER_SIZE)
if LOW_VRAM:
    paint.enable_model_cpu_offload()


def fix_material(mesh):
    image = mesh.visual.material.image
    mesh.visual.material = trimesh.visual.material.PBRMaterial(
        baseColorTexture=image, metallicFactor=0.0, roughnessFactor=0.9)
    return mesh


for asset in ASSETS:
    key = asset["key"]
    shape_path = os.path.join(SHAPE_DIR, f"{key}.glb")
    if not os.path.exists(shape_path):
        raise FileNotFoundError(f"Run block 6 to create {shape_path}.")
    out_glb = os.path.join(OUT_DIR, f"hunyuan3d2-{key}.glb")
    if os.path.exists(out_glb) and os.path.getmtime(out_glb) >= os.path.getmtime(shape_path):
        print("skip (texture is current):", out_glb)
        continue

    t0 = time.time()
    front_image = Image.open(os.path.join(SHAPE_DIR, f"{key}-input.png"))
    mesh = trimesh.load(shape_path, force="mesh")
    try:
        textured = paint(mesh, image=front_image)
    except torch.cuda.OutOfMemoryError:
        print(f"{key}: out of VRAM. Set RENDER_SIZE = 1024 in block 2, rerun blocks 2 and 7.")
        torch.cuda.empty_cache()
        continue
    textured = fix_material(textured)
    textured.export(out_glb)

    metadata = {
        "file": os.path.basename(out_glb),
        "source_image": os.path.basename(asset["source"]),
        "view_images": {"front": f"used-{key}-front-cutout.png", **{view: f"used-{key}-{view}.jpg" for view in ("left", "back", "right")}},
        "view_generator": "MV-Adapter",
        "view_base_model": VIEW_BASE_MODEL,
        "view_vae_model": VIEW_VAE_MODEL,
        "view_adapter_repo": VIEW_ADAPTER_REPO,
        "view_adapter_commit": VIEW_ADAPTER_COMMIT,
        "view_seed": VIEW_SEED,
        "view_azimuth_deg": VIEW_AZIMUTH_DEG,
        "shape_generator": "Tencent Hunyuan3D-2mv",
        "shape_model": SHAPE_MODEL,
        "shape_repo_commit": HUNYUAN_COMMIT,
        "shape_flashvdm": bool(SHAPE_FLASHVDM and "turbo" in SHAPE_MODEL),
        "paint_model": PAINT_MODEL,
        "shape_seed": SEED,
        "preset": PRESETS[asset["category"]],
        "faces": int(len(textured.faces)),
        "texture_size": TEXTURE_SIZE,
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    }
    with open(out_glb.replace(".glb", ".json"), "w") as file:
        json.dump(metadata, file, indent=2)
    print(f"{key}: {len(textured.faces):,} faces ({time.time() - t0:.0f} s) -> {out_glb}")
    gc.collect()
    torch.cuda.empty_cache()

del paint
gc.collect()
torch.cuda.empty_cache()
print("Stage 2 done.")

In [ ]:
# @title 8. Download and mark sources used
import glob
import os
import shutil
import time
from google.colab import files

ready_outputs = []
for asset in ASSETS:
    key = asset["key"]
    for extension in (".glb", ".json"):
        path = os.path.join(OUT_DIR, f"hunyuan3d2-{key}{extension}")
        if os.path.exists(path):
            ready_outputs.append(path)
            print(f"{os.path.basename(path):60s} {os.path.getsize(path) / 1024**2:6.1f} MB")

if not ready_outputs:
    raise FileNotFoundError("No completed GLB/metadata found. Run blocks 6 and 7 first.")

for path in ready_outputs:
    shutil.copy2(path, OUT_DRIVE_DIR)
print("Saved completed outputs to", OUT_DRIVE_DIR)

zip_base = f"/content/hunyuan3d2-{time.strftime('%Y%m%d-%H%M')}"
shutil.make_archive(zip_base, "zip", OUT_DIR)
files.download(zip_base + ".zip")

# Drive now contains the GLB, so consumed fronts can be marked without risking the only copy.
for asset in ASSETS:
    key = asset["key"]
    glb_on_drive = os.path.join(OUT_DRIVE_DIR, f"hunyuan3d2-{key}.glb")
    source = asset["source"]
    if not os.path.exists(glb_on_drive) or not os.path.exists(source):
        continue
    used_source = os.path.join(INPUT_DIR, f"used-{os.path.basename(source)}")
    if os.path.exists(used_source):
        raise FileExistsError(f"Refusing to overwrite existing source archive: {used_source}")
    os.replace(source, used_source)
    print("Marked source used:", os.path.basename(used_source))

print("Done. GLB is in Drive/out and in the download archive.")

## 出問題時

| 狀況 | 做法 |
|---|---|
| 區塊 1 回報 Python 版本不符 | 「執行階段 → 變更執行階段類型 → 執行階段版本」選 2026.07，再從區塊 1 開始 |
| 2026.07 已從選單消失 | Colab 只保留一段時間的舊版本；需改用新版 runtime 並更新區塊 1 的版本檢查與區塊 3 的 `PINNED` |
| 區塊 2 空間不足 | 清出 Drive 空間，預留約 22 GB |
| 區塊 3 安裝失敗 | 查看最後 30 行錯誤；安裝會保留 Colab 的 torch/numpy 版本 |
| 區塊 3 編譯的擴充失效 | 刪除 `DRIVE_ROOT/wheels/` 下對應的資料夾後，重開 runtime 重跑區塊 1、2、3 |
| 區塊 5 找不到模型或回報 401/403 | 確認已接受 Hugging Face 模型條款；需要時在 Colab Secrets 設 `HF_TOKEN` |
| 區塊 5 out of VRAM | 確認區塊 2 的 `VIEW_COUNT` 是 4，重開 runtime 後從區塊 1 跑 |
| 區塊 5 runtime 當掉（RAM 不足） | 確認未在同一 runtime 載入其他模型；重開 runtime 從區塊 1 跑 |
| 左、後、右視角不理想 | 改 `VIEW_SEED` 並重跑區塊 5；看完四張再決定是否執行區塊 6 |
| 區塊 6 out of VRAM | 選用 `hunyuan3d-dit-v2-mv-turbo`，並降低該類別在區塊 2 `PRESETS` 中的 `octree` |
| 區塊 6 FlashVDM 出錯 | 區塊 2 把 `SHAPE_FLASHVDM` 改成 False，重跑 2、6 |
| 區塊 7 out of VRAM | 區塊 2 把 `RENDER_SIZE` 改成 1024、`TEXTURE_SIZE` 改成 1024，重跑區塊 2 和 7 |
| 來源圖已處理 | `used-` 開頭的圖片會略過；放入新的 `<類別>-<id>-front.jpg` |
| 斷線或下載中斷 | 重開後從區塊 1 跑；已完成 GLB 在 `DRIVE_ROOT/out`，未標記的來源圖可再處理 |
| 背面造型或貼圖接縫不準 | 單張概念圖無法保證遮擋面一致；換視角種子，或在 Blender 裡修整 |